In [34]:
#Install Libraries
!pip -q install requests jsonschema

In [35]:
import os
import json
import re
import requests
import joblib
import pandas as pd

from jsonschema import validate, ValidationError

In [36]:
from getpass import getpass

api_key = getpass("Enter OpenRouter API Key: ")

os.environ["LLM_API_KEY"] = api_key

Enter OpenRouter API Key: ··········


In [37]:
# Load the Model
best_model = joblib.load("best_model.pkl")

print("Best Model Loaded Successfully")
print(type(best_model))

Best Model Loaded Successfully
<class 'sklearn.pipeline.Pipeline'>


In [38]:
# Build the Reusable call_llm() Function
# ==========================================
# OpenRouter Configuration
# ==========================================

API_URL = "https://openrouter.ai/api/v1/chat/completions"

MODEL_NAME = "meta-llama/llama-3.1-8b-instruct"

In [39]:
# ==========================================
# Reusable LLM Function
# ==========================================

def call_llm(system_prompt,
             user_prompt,
             temperature=0.0,
             max_tokens=512):

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    response = requests.post(
        API_URL,
        headers=headers,
        json=payload
    )

    if response.status_code != 200:
        print("Error:", response.status_code)
        print(response.text)
        return None

    return response.json()["choices"][0]["message"]["content"]

In [40]:
# Test the API
system_prompt = "You are a helpful assistant."

user_prompt = "Reply with only the word: hello"

response = call_llm(
    system_prompt,
    user_prompt,
    temperature=0
)

print(response)

Hello


In [41]:
# ==========================================
# PII Detection Function
# ==========================================

import re

def has_pii(text):

    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'

    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'

    return bool(
        re.search(email_pattern, text)
        or
        re.search(phone_pattern, text)
    )

In [42]:
# ==========================================
# Guardrail Test 1
# ==========================================

user_input = "My email is student@gmail.com"

if has_pii(user_input):

    print("Input blocked: PII detected.")

else:

    print(call_llm(
        "You are a helpful assistant.",
        user_input
    ))

Input blocked: PII detected.


In [43]:
# ==========================================
# Guardrail Test 2
# ==========================================

user_input = "Explain what machine learning is."

if has_pii(user_input):

    print("Input blocked: PII detected.")

else:

    response = call_llm(
        "You are a helpful assistant.",
        user_input
    )

    print(response)

Machine learning is a subset of artificial intelligence (AI) that involves training algorithms to learn from data, allowing them to make predictions, classify objects, or make decisions without being explicitly programmed.

In traditional programming, a computer is given a set of rules and instructions to follow. In contrast, machine learning algorithms are designed to improve their performance on a task over time, based on the data they are trained on. This is achieved through a process called "training," where the algorithm is fed a large dataset and learns to identify patterns, relationships, and trends within that data.

There are several key aspects of machine learning:

1. **Data**: Machine learning algorithms require large amounts of data to learn from. This data can be in the form of text, images, audio, or any other type of data that can be represented numerically.
2. **Algorithms**: Machine learning algorithms are designed to analyze the data and make predictions or decisions

# PII Guardrail

A regular-expression-based guardrail was implemented before every LLM request.

The guardrail checks user input for:

- Email addresses
- Phone numbers

If any personally identifiable information (PII) is detected, the request is blocked before reaching the LLM.

Two test cases were executed:

1. Input containing an email address (Blocked)
2. Input without PII (Allowed)

# Task 3 – Create JSON Schema

In [44]:
# ==========================================
# JSON Schema for LLM Explanation
# ==========================================

prediction_schema = {
    "type": "object",
    "properties": {
        "prediction_label": {
            "type": "string"
        },
        "confidence_level": {
            "type": "string"
        },
        "top_reason": {
            "type": "string"
        },
        "second_reason": {
            "type": "string"
        },
        "next_step": {
            "type": "string"
        }
    },

    "required": [
        "prediction_label",
        "confidence_level",
        "top_reason",
        "second_reason",
        "next_step"
    ]
}

print("Schema Created Successfully")

Schema Created Successfully


## JSON Schema

A JSON schema was defined to validate every explanation returned by the LLM.

The schema contains five required scalar fields:

- prediction_label
- confidence_level
- top_reason
- second_reason
- next_step

Each LLM response must conform to this schema before being accepted.

# Task 4 – Create Three Sample Inputs

In [45]:
df= pd.read_csv("encoded_features.csv")

print(df.shape)

(10000, 8444)


In [46]:
# ==========================================
# Select Three Test Inputs
# ==========================================

sample_inputs = df.iloc[[5, 20, 100]].copy()

sample_inputs.head()

,CaseOrder,Zip,Lat,Lng,Population,Children,Age,Education,Income,VitD_levels,...,Arthritis_Yes,Diabetes_Yes,Hyperlipidemia_Yes,BackPain_Yes,Allergic_rhinitis_Yes,Reflux_esophagitis_Yes,Asthma_Yes,Services_CT Scan,Services_Intravenous,Services_MRI
5,6,74423,35.67302,-95.19180,981,1.0,76.0,3,33942.28,19.956143,...,True,True,False,True,True,False,False,False,False,False
20,21,55103,44.96425,-93.12449,13947,2.0,53.0,3,8686.47,19.968337,...,False,False,False,False,False,True,False,False,False,False
100,101,63546,40.33071,-92.51474,1950,0.0,34.0,1,33942.28,17.014356,...,False,False,True,False,False,False,False,False,True,False


In [47]:
# ==========================================
# Predict Class
# ==========================================

predictions = best_model.predict(df)

probabilities = best_model.predict_proba(df)

print(predictions)

print()

print(probabilities)

[0 0 0 ... 1 1 1]

[[0.975 0.025]
 [0.95  0.05 ]
 [0.965 0.035]
 ...
 [0.21  0.79 ]
 [0.145 0.855]
 [0.065 0.935]]


## Task 4 – Prompt Engineering (Track C)

In [48]:
# ==========================================
# System Prompt
# ==========================================

system_prompt = """
You are an AI assistant that explains machine learning predictions.

Return ONLY valid JSON.

prediction_label should be one of:
- "Readmission"
- "No Readmission"

confidence_level should be one of:
- "Low"
- "Medium"
- "High"

Return exactly this JSON structure:

{
  "prediction_label":"",
  "confidence_level":"",
  "top_reason":"",
  "second_reason":"",
  "next_step":""
}
"""

In [49]:
# ==========================================
# Create User Prompt
# ==========================================

def create_user_prompt(features, prediction, probability):

    label = "Readmission" if prediction == 1 else "No Readmission"

    return f"""
Patient Feature Values:

{features.to_dict()}

Machine Learning Prediction:

Prediction Label: {label}

Prediction Probability:

{probability:.4f}

Based on these values, explain the prediction.

Return ONLY valid JSON.
"""

In [50]:
# ==========================================
# Test Prompt
# ==========================================

sample = df.iloc[0]

prediction = predictions[0]

probability = probabilities[0].max()

user_prompt = create_user_prompt(
    sample,
    prediction,
    probability
)

print(user_prompt)


Patient Feature Values:

{'CaseOrder': 1, 'Zip': 35621, 'Lat': 34.3496, 'Lng': -86.72508, 'Population': 2951, 'Children': 1.0, 'Age': 53.0, 'Education': 5, 'Income': 86575.93, 'VitD_levels': 17.80233049, 'Doc_visits': 6, 'Full_meals_eaten': 0, 'VitD_supp': 0, 'Complication_risk': 1, 'Overweight': 0.0, 'Anxiety': 1.0, 'Initial_days': 10.58576971, 'Additional_charges': 17939.40342, 'Item1': 3, 'Item2': 3, 'Item3': 2, 'Item4': 2, 'Item5': 4, 'Item6': 3, 'Item7': 3, 'Item8': 4, 'City_Abbeville': False, 'City_Aberdeen': False, 'City_Abilene': False, 'City_Abingdon': False, 'City_Abita Springs': False, 'City_Absecon': False, 'City_Accokeek': False, 'City_Accord': False, 'City_Adams': False, 'City_Addieville': False, 'City_Addington': False, 'City_Addison': False, 'City_Addyston': False, 'City_Admire': False, 'City_Adolphus': False, 'City_Adrian': False, 'City_Afton': False, 'City_Agness': False, 'City_Agoura Hills': False, 'City_Agra': False, 'City_Aguas Buenas': False, 'City_Aguila': False

In [51]:
# ==========================================
# LLM Prediction Explanation
# ==========================================

response = call_llm(
    system_prompt,
    user_prompt,
    temperature=0
)

print(response)

{
  "prediction_label": "No Readmission",
  "confidence_level": "High",
  "top_reason": "Patient's age and education level are within normal limits, and there are no significant comorbidities.",
  "second_reason": "Patient's initial days in the hospital are within the normal range, and there are no signs of complications.",
  "next_step": "Patient can be discharged from the hospital with regular follow-up appointments."
}


In [52]:
# ==========================================
# Parse and Validate JSON Response
# ==========================================

try:

    response = response.strip()

    parsed = json.loads(response)

    validate(
        instance=parsed,
        schema=prediction_schema
    )

    status = "Pass"

    print("Validation Passed")
    print(parsed)

except json.JSONDecodeError as e:

    print("JSON Decode Error:")
    print(e)

    parsed = {
        "prediction_label": None,
        "confidence_level": None,
        "top_reason": None,
        "second_reason": None,
        "next_step": None
    }

    status = "Fail"

except ValidationError as e:

    print("Schema Validation Error:")
    print(e)

    parsed = {
        "prediction_label": None,
        "confidence_level": None,
        "top_reason": None,
        "second_reason": None,
        "next_step": None
    }

    status = "Fail"

print("\nValidation Status :", status)

Validation Passed
{'prediction_label': 'No Readmission', 'confidence_level': 'High', 'top_reason': "Patient's age and education level are within normal limits, and there are no significant comorbidities.", 'second_reason': "Patient's initial days in the hospital are within the normal range, and there are no signs of complications.", 'next_step': 'Patient can be discharged from the hospital with regular follow-up appointments.'}

Validation Status : Pass


## Prompt Engineering

A zero-shot prompting strategy was used.

The system prompt instructs the LLM to behave as a machine learning prediction explanation assistant and to return only valid JSON.

The user prompt contains:

- Feature values
- Predicted class
- Prediction probability

Temperature was set to **0** to ensure deterministic and reproducible structured outputs suitable for schema validation.

In [53]:
# ==========================================
# Store Results
# ==========================================

results = []

In [54]:
# ==========================================
# Run Pipeline for 3 Samples
# ==========================================

for i in range(3):

    # -------------------------------
    # Get Sample
    # -------------------------------
    sample = df.iloc[i]

    prediction = predictions[i]

    probability = probabilities[i].max()

    # -------------------------------
    # Create Prompt
    # -------------------------------
    user_prompt = create_user_prompt(
        sample,
        prediction,
        probability
    )

    # -------------------------------
    # Guardrail Check
    # -------------------------------
    if has_pii(user_prompt):

        print(f"\nSample {i+1}: Blocked (PII Detected)\n")

        results.append({
            "Sample": i+1,
            "Prediction": prediction,
            "Probability": round(probability,4),
            "Validation":"Blocked"
        })

        continue

    # -------------------------------
    # Call LLM
    # -------------------------------
    response = call_llm(
        system_prompt,
        user_prompt,
        temperature=0
    )

    print(f"\n========== SAMPLE {i+1} ==========")

    print("\nRaw LLM Response:\n")

    print(response)

    # -------------------------------
    # Parse JSON
    # -------------------------------
    try:

        parsed = json.loads(response.strip())

        validate(
            instance=parsed,
            schema=prediction_schema
        )

        status = "Pass"

    except Exception as e:

        print(e)

        parsed = {
            "prediction_label": None,
            "confidence_level": None,
            "top_reason": None,
            "second_reason": None,
            "next_step": None
        }

        status = "Fail"

    print("\nValidation :", status)

    results.append({

        "Sample": i+1,

        "Prediction": prediction,

        "Probability": round(probability,4),

        "Validation": status

    })


========== SAMPLE 1 ==========

Raw LLM Response:

{
  "prediction_label": "No Readmission",
  "confidence_level": "High",
  "top_reason": "Low risk of readmission due to low probability of complications",
  "second_reason": "Patient's age and health status indicate a low risk of readmission",
  "next_step": "Discharge the patient with regular follow-up appointments"
}

Validation : Pass

========== SAMPLE 2 ==========

Raw LLM Response:

{
  "prediction_label": "No Readmission",
  "confidence_level": "High",
  "top_reason": "Low risk of readmission due to low probability of complications and high probability of successful treatment",
  "second_reason": "Patient's age and education level are within normal ranges, indicating a lower risk of readmission",
  "next_step": "Continue with standard treatment plan and monitor patient's progress closely"
}

Validation : Pass

========== SAMPLE 3 ==========

Raw LLM Response:

{
  "prediction_label": "No Readmission",
  "confidence_level": "Hig

In [55]:
# ==========================================
# Summary Table
# ==========================================

results_df = pd.DataFrame(results)

results_df

,Sample,Prediction,Probability,Validation
0,1,0,0.975,Pass
1,2,0,0.950,Pass
2,3,0,0.965,Pass


# Three Input Demonstration

The complete prediction explanation pipeline was executed for three different feature vectors.

For each sample:

- Machine learning prediction was generated.
- Prediction probability was calculated.
- Feature values were converted into a prompt.
- The prompt was checked using the PII guardrail.
- The LLM generated a structured JSON explanation.
- The JSON output was validated against the predefined schema.

In [57]:
# ==========================================
# Temperature Comparison
# ==========================================

comparison_results = []

for i in range(3):

    sample = df.iloc[i]

    prediction = predictions[i]

    probability = probabilities[i].max()

    user_prompt = create_user_prompt(
        sample,
        prediction,
        probability
    )

    # Temperature = 0
    response_temp0 = call_llm(
        system_prompt,
        user_prompt,
        temperature=0
    )

    # Temperature = 0.7
    response_temp07 = call_llm(
        system_prompt,
        user_prompt,
        temperature=0.7
    )

    comparison_results.append({

        "Input": f"Sample {i+1}",

        "Output at Temp=0": response_temp0,

        "Output at Temp=0.7": response_temp07,

        "Key Difference": "Temp=0 is deterministic; Temp=0.7 may vary."

    })

In [58]:
comparison_df = pd.DataFrame(comparison_results)

comparison_df

,Input,Output at Temp=0,Output at Temp=0.7,Key Difference
0,Sample 1,"{\n ""prediction_label"": ""No Readmission"",\n ...","{\n ""prediction_label"": ""No Readmission"",\n ...",Temp=0 is deterministic; Temp=0.7 may vary.
1,Sample 2,"{\n ""prediction_label"": ""No Readmission"",\n ...","{\n ""prediction_label"": ""No Readmission"",\n ...",Temp=0 is deterministic; Temp=0.7 may vary.
2,Sample 3,"{\n ""prediction_label"": ""No Readmission"",\n ...","{\n ""prediction_label"": ""No Readmission"",\n ...",Temp=0 is deterministic; Temp=0.7 may vary.
